# PS2: Right-Party Contact Prediction (Perfect Pipeline)

This pipeline predicts Right-Party Contact by utilizing time-decay network responses and aggregate call behaviors derived from our deep EDA.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score
import os

## 2. Data Loading

In [ ]:
base_path = r"../../Dataset/data"

In [ ]:
phones = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "phones.csv"))
dial_attempts = pd.read_csv(os.path.join(base_path, "shared", "dial_attempts.csv"))
verified = pd.read_csv(os.path.join(base_path, "ps2_right_party_contact", "verified_contact_points.csv"))

## 3. Feature Engineering with Time-Decay

In [ ]:
verified['is_rpc'] = (verified['verified_status'] == 'borrower_number').astype(int)
dial_attempts['attempt_ts'] = pd.to_datetime(dial_attempts['attempt_ts'])

In [ ]:
# Sort by time to calculate recency
dial_attempts = dial_attempts.sort_values('attempt_ts')
anchor_date = pd.to_datetime('2026-08-01')

In [ ]:
# Aggregate with ratios as dictated by EDA
attempts_agg = dial_attempts.groupby('phone_id').agg(
    total_attempts=('attempt_id', 'count'),
    switched_off_count=('network_response', lambda x: (x == 'switched_off').sum()),
    answered_count=('talk_duration_s', lambda x: (x > 0).sum()),
    last_attempt_date=('attempt_ts', 'max'),
    avg_talk_time=('talk_duration_s', 'mean')
).reset_index()

In [ ]:
attempts_agg['switched_off_ratio'] = attempts_agg['switched_off_count'] / attempts_agg['total_attempts']
attempts_agg['answered_ratio'] = attempts_agg['answered_count'] / attempts_agg['total_attempts']
attempts_agg['days_since_last_attempt'] = (anchor_date - attempts_agg['last_attempt_date']).dt.days

In [ ]:
data = pd.merge(verified, phones, on='phone_id', how='left')
data = pd.merge(data, attempts_agg, on='phone_id', how='left').fillna(
    {'total_attempts':0, 'switched_off_ratio':0, 'answered_ratio':0, 'days_since_last_attempt':-1, 'avg_talk_time': 0}
)

In [ ]:
features = ['total_attempts', 'switched_off_ratio', 'answered_ratio', 'days_since_last_attempt', 'avg_talk_time', 'priority_slot']
X = data[features]
y = data['is_rpc']

## 4. Modeling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

In [ ]:
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")